# Week 7, Day 5: Judging your fine-tuned pricer

**Lesson:** [the Colab notebook](https://colab.research.google.com/drive/16e8aY_BlHjzzcR-2dCyDMCPdOQ8XeN1e) linked from [week7/day5.ipynb](../../../week7/day5.ipynb), *Evaluation*. **About 20 minutes.**

**What you practice here**

- Loading a fine-tuned adapter from the Hub onto the 4-bit base with `PeftModel.from_pretrained`, pinned to one checkpoint of the run by its revision.
- Handing the course's evaluator what it calls: a predictor of one datapoint, the lesson's `model_predict`, now on the fine-tuned model.
- Judging the predictions in a way you can repeat: `set_seed`, then the course's `evaluate`, set beside the base model and the frontier models.

**How this notebook works.** Run the first code cell once. Then, for each part below, write it in its cell, run that cell, and run the check under it. The checks hand your code stand-ins shaped like the real client and libraries: they record what your code sends and answer with canned replies, so nothing is called and nothing is spent. Write from memory, and look back at the lesson only when you are stuck.

When every check passes, save the notebook, run the checker and push, as in every session. The tracker then marks the lesson done.

In [ ]:
# Run this cell first. It finds plan/check.py in your fork and loads this exercise's checks.
import os, pathlib, sys
ROOT = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "plan" / "check.py").is_file()), None)
if ROOT is None and ("google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ):
    # in Colab: fetch the checker from your fork on GitHub, and the packages Colab lacks
    import subprocess, urllib.request
    ROOT = pathlib.Path("/content/fork")
    (ROOT / "plan").mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve("https://raw.githubusercontent.com/starlest/llm_engineering/main/plan/check.py", ROOT / "plan" / "check.py")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openai", "python-dotenv", "tiktoken"], check=False)
    print("In Colab, the checks run here too. To keep your work, use File > Save a copy in GitHub, to this notebook's path in your fork.")
if ROOT:
    sys.path.insert(0, str(ROOT / "plan"))
    try:
        from check import exercise
    except ImportError:
        raise ImportError("your plan/check.py is older than this notebook: run git pull, or copy the new one from the tracker page") from None
    check = exercise("week7/day5", globals())
else:  # outside your fork: write and try the parts here, and run the checks in your codespace
    def check(part=None):
        print("The checks are off here, as plan/check.py is not in a folder above this notebook. Run them in your codespace.")
    print("plan/check.py was not found above this notebook, so the checks are off here.")

## The task

Day 5 is the moment of truth. The lesson loads the 4-bit base model, puts the fine-tuned adapter on it, pinned to one
checkpoint of the training run, and judges its predictions with the course's `evaluate`. The errors to beat are a
human's $87.62 and GPT-4.1-nano's $62.51. Write the loading and the judging as functions. Then use them to judge the
lesson's fine-tune, or your own trial run from Days 3 and 4, and to price some wrestling merchandise.

transformers and peft are not in the checks' Python, so `load_finetuned` takes the two classes it uses as
parameters, and `make_predictor` takes the model and the tokenizer. `score` takes `evaluate` and `set_seed` as
parameters too, so the checks can see what it hands them. The checks give your functions stand-ins: a loader, an
adapter loader that knows which checkpoints exist, and a model whose `generate` answers with a canned price.

## What each call should give

```python
fine_tuned = load_finetuned(AutoModelForCausalLM, PeftModel, quant_config, "ed-donner/price-2025-11-28_18.47.07",
                            revision="b19c8bfea3b6ff62237fbb0a8da9779fc12cefbd")
# Llama 3.2 3B in 4 bits, with the full run's adapter as it was at that checkpoint

model_predict = make_predictor(fine_tuned, tokenizer)
model_predict({"prompt": "What does this cost to the nearest dollar?\n\nTitle: WWE Championship Replica Title Belt\n"
                         "Category: Collectibles\nBrand: WWE\nDescription: A full-size replica of the WWE "
                         "Championship title belt.\nDetails: Zinc-alloy plates on a genuine leather strap, with a "
                         "carry bag.\n\nPrice is $", "completion": "499.99"})
# '499.00<|end_of_text|>': the new tokens only

score(evaluate, set_seed, model_predict, test)
# seeds with 42, then runs the course's evaluate on the first 200 test datapoints: its charts and the error
```

In [ ]:
# The imports for this exercise. Add more here if you need them: the checks have the standard library, openai, pydantic, dotenv, numpy, bs4 and tiktoken.

### 1 · `load_finetuned(auto_model, peft_model, quant_config, adapter_id, revision=None, base="meta-llama/Llama-3.2-3B")`

Loads the base model and puts the fine-tuned adapter on it, pinned to a checkpoint when a revision is given.
`auto_model` is `AutoModelForCausalLM`, `peft_model` is peft's `PeftModel`, and `quant_config` is a ready
`BitsAndBytesConfig`. Load the model `base` with `quant_config` as its quantization config, and let the loader place
it on the GPU, as the lesson's `device_map="auto"` does. Then load the adapter `adapter_id` onto it with the loader
that `PeftModel` has: at `revision` when one is given, and at its latest version when `revision` is `None`, as the
lesson does. Return the model that the adapter loader gives.

A revision is the id of one commit of the adapter's repo. The trainer pushed a commit at each checkpoint, so a
revision picks the adapter as it was at that point of the run. Pass it by name: the loader's third parameter is the
adapter's name, not its revision.

<details><summary>Hint</summary>

```python
base_model = auto_model.from_pretrained(base, quantization_config=quant_config, device_map="auto")
peft_model.from_pretrained(base_model, adapter_id, revision=revision)  # or without revision=
```

</details>

In [ ]:
def load_finetuned(auto_model, peft_model, quant_config, adapter_id, revision=None, base="meta-llama/Llama-3.2-3B"):
    # Write load_finetuned here, as described above.
    raise NotImplementedError("load_finetuned is not written yet")

In [ ]:
check("load_finetuned")

### 2 · `make_predictor(model, tokenizer, device="cuda", max_new_tokens=8)`

Makes the predictor that the course's `evaluate` calls: a function of one datapoint that returns the model's new
text. Return a function `model_predict(datapoint)`, and make no call to the model in `make_predictor` itself.
`model_predict` takes a test datapoint, a dict such as `{"prompt": "...Price is $", "completion": "129.99"}`. It
tokenizes the datapoint's `"prompt"` as PyTorch tensors, moves them to `device`, and generates at most
`max_new_tokens` new tokens, passing the attention mask along with the ids. It returns the decoded text of the new
tokens only, from the first row. This is Day 2's code, now on the fine-tuned model; as there, you may leave out the
lesson's `with torch.no_grad():`.

<details><summary>Hint</summary>

```python
def model_predict(datapoint):
    inputs = tokenizer(datapoint["prompt"], return_tensors="pt").to(device)
    output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens)
    return tokenizer.decode(output_ids[0, inputs["input_ids"].shape[1]:])
```

</details>

In [ ]:
def make_predictor(model, tokenizer, device="cuda", max_new_tokens=8):
    # Write make_predictor here, as described above.
    raise NotImplementedError("make_predictor is not written yet")

In [ ]:
check("make_predictor")

### 3 · `score(evaluate, set_seed, predictor, data, seed=42, size=200)`

Judges a predictor's predictions in a way you can repeat: it seeds the random numbers, then runs the course's
evaluation. `evaluate` is the one in the course's `util.py`, and `set_seed` is transformers'. Seed with `seed`
first. Then evaluate `predictor` on `data`, over `size` datapoints, and return what the evaluation returns. The seed
fixes any random sampling in `generate`, so a second run gives the same predictions and the same error.

<details><summary>Hint</summary>

```python
set_seed(seed)
return evaluate(predictor, data, size=size)
```

</details>

In [ ]:
def score(evaluate, set_seed, predictor, data, seed=42, size=200):
    # Write score here, as described above.
    raise NotImplementedError("score is not written yet")

In [ ]:
check("score")

## Check everything

The cell below runs all 7 checks again. When they all pass, save this notebook (Ctrl+S, or Cmd+S on a Mac), then run the checker and push, as in every session:

```bash
uv run python plan/check.py
git add -A
git commit -m "Week 7, Day 5 exercise"
git push
```

The checker runs the same checks on the saved notebook, and the tracker marks the lesson done. It reads only your imports, functions, classes and constants, so the experiments in your other cells never get in the way.

In [ ]:
check()

## Try it for real

Now run what you wrote on the real thing. This needs a GPU, so it runs in Google Colab: push this notebook, then [open it in Colab](https://colab.research.google.com/github/starlest/llm_engineering/blob/main/plan/exercises/week7/day5.ipynb), choose a T4 runtime, and run the cells from the top. The first cell fetches the checker from your fork, so the checks run in Colab too. Write the steps in the 2 cells at the end.

Colab works on its own copy of the notebook. To keep the Try code you write there, use **File > Save a copy in GitHub** (to this notebook's path in your fork), then run `git pull` in your codespace before you commit there again. Your parts are already in your fork, which is what the tracker reads.

Colab's free T4 is enough. Add your Hugging Face token to Colab's secrets as `HF_TOKEN`; it also opens your own
private adapter, if you trained one on Days 3 and 4. Judging 200 test items takes a few minutes.

1. Keep the `!pip install` and `!wget` lines in the first Try cell: they fetch bitsandbytes and the course's
   `util.py`. Log in to Hugging Face with your `HF_TOKEN` secret.
2. Build the 4-bit config, with bf16 compute when `torch.cuda.get_device_capability()[0] >= 8` and fp16 otherwise,
   and the tokenizer with its pad token, as the lesson does.
3. Load the lesson's lite adapter, `ed-donner/price-2025-11-30_15.10.55-lite`, with `load_finetuned`, as
   `fine_tuned`, and set its `generation_config.pad_token_id`. You can instead load the full run,
   `ed-donner/price-2025-11-28_18.47.07`, at the lesson's revision, or your own trial adapter from Days 3 and 4.
4. Make `model_predict = make_predictor(fine_tuned, tokenizer)`, and price a wrestling item of your own, written in
   the format the model learned: the question, then five lines (Title, Category, Brand, Description and Details),
   then `Price is $`.
5. Load the test split of `ed-donner/items_prompts_lite` as `test`, import `evaluate` from `util` and `set_seed`
   from transformers, and judge the fine-tune with `score(evaluate, set_seed, model_predict, test)`. Set its error
   beside those in `week7/results.ipynb`: 110.72 for the base model, 65.40 for the lite run and 39.85 for the full
   one.

<details><summary>Stuck? One way to write it</summary>

```python
!pip install -q --upgrade bitsandbytes trl
!wget -q https://raw.githubusercontent.com/ed-donner/llm_engineering/main/week7/util.py -O util.py
import torch
from google.colab import userdata
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

login(userdata.get("HF_TOKEN"), add_to_git_credential=True)
BASE_MODEL = "meta-llama/Llama-3.2-3B"
use_bf16 = torch.cuda.get_device_capability()[0] >= 8
quant_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4",
                                  bnb_4bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

fine_tuned = load_finetuned(AutoModelForCausalLM, PeftModel, quant_config, "ed-donner/price-2025-11-30_15.10.55-lite")
# Or the full run, at the lesson's checkpoint:
# fine_tuned = load_finetuned(AutoModelForCausalLM, PeftModel, quant_config, "ed-donner/price-2025-11-28_18.47.07",
#                             revision="b19c8bfea3b6ff62237fbb0a8da9779fc12cefbd")
fine_tuned.generation_config.pad_token_id = tokenizer.pad_token_id
print(f"Memory footprint: {fine_tuned.get_memory_footprint() / 1e6:,.1f} MB")
```

```python
from datasets import load_dataset
from transformers import set_seed
from util import evaluate

model_predict = make_predictor(fine_tuned, tokenizer)
belt = {"prompt": "What does this cost to the nearest dollar?\n\nTitle: WWE Championship Replica Title Belt\n"
                  "Category: Collectibles\nBrand: WWE\nDescription: A full-size replica of the WWE Championship "
                  "title belt.\nDetails: Zinc-alloy plates on a genuine leather strap, with a carry bag.\n\n"
                  "Price is $"}
print(model_predict(belt))

test = load_dataset("ed-donner/items_prompts_lite", split="test")
score(evaluate, set_seed, model_predict, test)  # results.ipynb: base 110.72, lite 65.40, full 39.85
```

</details>

In [ ]:
!pip install -q --upgrade bitsandbytes trl
!wget -q https://raw.githubusercontent.com/ed-donner/llm_engineering/main/week7/util.py -O util.py
# Steps 1 to 3: log in, then load the 4-bit base model with the fine-tuned adapter on it

In [ ]:
# Steps 4 and 5: price a wrestling item, then judge the fine-tune with score